In [ ]:
import json
import pandas as pd
from tqdm import tqdm

In [ ]:
def generate_prompt_JD_prediction(query_table_name, query_table_description, query_table_column_names, query_table_column_descriptions,
                                  candidate_table_name, candidate_table_description, candidate_table_column_names, candidate_table_column_descriptions):

    system_msg = """
            You are given eight inputs: the target table name, the target table description, the target table columns names, the target table 
            column descriptions, the candidate table name, the candidate table description, all candidate table columns names and all 
            candidate table columns descriptions.

            Your task is to classify if the two tables are contextually similar, it means their descriptions are related or talk about similar topics,
            then if they do identify attributes with similar semantics so that they could be used to join the two tables. Answer only with the appropriate
            information in the specified JSON format.
            
            Definition of Joinability In relational databases, a JOIN combines rows from two tables based on matching column values that 
            represent the same real-world entity.

            Two tables are considered joinable if they contain at least one column (or semantically equivalent columns) whose values can be 
            matched.
            
            Instructions
            Carefully read the target table name.
            Carefully read the target table description.
            Carefully read the target table columns names.
            Carefully read the target table columns description.
            Carefully read the candidate table name.
            Carefully read the candidate table description.
            Carefully read the candidate table columns names.
            Carefully read the candidate table columns descriptions.
            Determine if the two tables exist within the same contextual domain. If they clearly belong to different domains, they are not 
            joinable, even if column names appear similar.
            Identify whether there is evidence of at least one shared or semantically equivalent column from the candidate table with the target 
            table columns.
            Answer with the most appropriate pair of columns found in the target table and candidate table.
            
            Answer only with:
            {  "table_joinable" : "Yes" | "No", 
                "column_joinable": "Yes" | "No",
                "target_match" : [target_column_names] | null
                "candidate_match": [candidate_column_names] | null}
               
            The output must be a single JSON object with exactly four fields:

                table_joinable - a string that must be "Yes" or "No".
                    "Yes" if both tables exists within the same contextual domain.
                    "No" if the tables are not contextually related.
            
                column_joinable – a string that must be "Yes" or "No".       
                    "Yes" if at least one column in the candidate table matches or semantically corresponds to a target table's column.
                    "No" if no column in the candidate table refers to the same entity any of the target table columns.

                target_match - the target table column names that are semantically equivalent to a candidate table columns.
                    If column_joinable is "Yes", specify the exact name of the columns in the target table that matches the candidate table column.
                    If column_joinable is "No", return null.
            
                candidate_match – the candidate table column names that are semantically equivalent to the target table columns.
                    If column_joinable is "Yes", specify the exact name of the columns in the candidate table that matches the target table column.
                    If column_joinable is "No", return null.
            
            Example (valid output formats):
                {"table_joinable": "Yes", "column_joinable" : "Yes", "target_match" : "user_id", "candidate_match": "USER_ID"}
                {"table_joinable": "Yes", "column_joinable" : "No", "target_match" : "null", "candidate_match": "null"}
                {"table_joinable": "No", "column_joinable" : "No", "target_match" : "null", "candidate_match": "null"}

            Example 01:
                  Target Table Name: SCS_Staff_Salaries_data_30th_June 2010.csv
                  Target Table Description: The table ""SCS_Staff_Salaries_data_30th_June 2010.csv"" contains information on the salaries of staff members at Ofgem as of June 30th, 2010, detailing individual names, job grades, titles (such as Partner, Director, Managing Director), full-time equivalency (FTE), parent department, organization, unit, as well as the actual pay range specifying both the floor and ceiling salaries, with additional notes where applicable.
                  Target Table Column Names: [Name, Parent Department, Organisaton]
                  Target Table Columns Descriptions: ["The ""Name"" column in the table represents the full names of individuals who hold various positions within the organization, Ofgem. The names listed denote employees at different grades and roles, including positions such as Partner, Director, Managing Director, Senior Partner, Secretariat, Group Finance Director, and Chief Executive. Most names are complete, but there is one instance where the name entry is missing, indicated by 'nan'. This column serves as an identifier for individuals, linking them to their respective grade, job title, department, and salary range, among other details.", "The ""Parent Department"" column indicates the overarching department or organizational unit to which each individual's job position belongs. In the context of this data, it consistently displays ""Ofgem"" for all entries, suggesting that all employees listed are part of or associated with the larger organizational entity known as Ofgem. This column helps to establish a clear organizational hierarchy and structure, affirming that regardless of an individual's specific job title or role, they are congregatively part of the main department, Ofgem. This uniformity in the column values underscores a centralized managerial or administrative system within which individual units like ""Ofgem Policy,"" ""Ofgem E-Serve,"" and ""Ofgem Group"" operate as sub-departments or subdivisions. Thus, the ""Parent Department"" column provides a means to relate and group job roles and employees' data by their primary affiliation to Ofgem, enabling insights into how different roles and employees fit within the broader organizational framework.", "The ""Organisaton"" column in the table consistently refers to ""Ofgem,"" indicating that all the individuals listed are associated with the same organization. The uniformity of this value across all entries suggests that this dataset is specific to a single entity, namely Ofgem, which is a regulatory authority for the electricity and downstream natural gas markets in Great Britain. Despite the various job titles, grades, departments, and pay levels of the individuals, their affiliation with Ofgem remains constant, reflecting that the dataset's scope is limited to this particular organization. The column does not provide any additional differentiation or detail about the organization beyond its name."]
                  Candidate Table Name: ability_ids.csv
                  Candidate Table Descriptions: The table ""ability_ids.csv"" contains a list of abilities in a game, with each entry representing a unique ability through a combination of an ""ability_id"" and corresponding ""ability_name"", detailing specific abilities such as ""antimage_blink"" and ""juggernaut_omni_slash"" as well as unique identifiers that may be utilized for game mechanics or reference.
                  Candidate Table JOIN Keys: [ability_id]
                  Candidate Table Column Descriptions: ["The target column, ""ability_id,"" is a numerical identifier that uniquely distinguishes each ability associated with characters or units within a specific context, likely a gaming environment. The values in this column appear to be sequential integers beginning with 0 and continuing into the thousands, such as 5001, 5002, and so on. Each unique number correlates directly with a specific ability name listed in the adjacent column, ""ability_name."" This structured format enables efficient referencing and management of abilities within a database, allowing developers or data analysts to quickly retrieve, manipulate, or update information related to a particular ability by using these unique numeric identifiers. The consistent format across all entries suggests an organized system for cataloging a wide range of abilities, potentially used for programming logic, data tracking, or enhancing user interface functionality within the application or game in which these abilities are employed."]
                  Answer: {"table_joinable": "No", "column_joinable" : "No", "target_match" : "null", "candidate_match": "null"}

            Example 02:
                  Target Table Name: animal-control-inventory-lost-and-found.csv
                  Target Table Description: The table ""animal-control-inventory-lost-and-found.csv"" contains detailed information about lost and found animals, specifically documenting the breed, color, specific and creation dates of the record, the animal's name and sex, and the state of the record, associated with the animals' status as either lost, found, or matched with their owners.
                  Target Table Column Names: [Date, DateCreated, Name]
                  Target Table Column Descriptions: ["The ""Date"" column in the provided table represents the specific date associated with each record of a pet. It indicates an event date that is relevant to the particular animal listed in the dataset. This date appears to be recorded in the format YYYY-MM-DD and serves as an indicator of when a particular situation, such as a pet being lost, matched, or found, occurred. The variation in dates, which range from as early as 1999 to as late as 2020, suggests it may be capturing the moment an event first happened for each pet. There seems to be a connection between this column and the ""State"" column, revealing the timing of when the pet's status, whether lost, found, or matched, was updated. Overall, this column is integral for understanding the timeline and historical context surrounding each pet entry within the table.", "The ""DateCreated"" column represents the date on which a record was entered or created in the system. The format of the date is the standard ""YYYY-MM-DD"" format, indicating the year, month, and day respectively. The values in this column often coincide with the ""Date"" column when the event and its logging or record entry occurred on the same day; however, it is possible for these dates to differ, as seen in some examples within the dataset, where events that occurred on certain past dates were only entered into the system much later, suggesting that the ""DateCreated"" column is exclusively focused on the timestamp marking the creation of the record itself, rather than the actual date of the event being described.", "The 'Name' column in the table represents the given names of animals that are the subject of this data. These names may be names assigned by their owners, shelter staff, or other individuals responsible for the animal. The names appear to vary greatly, showcasing a range of personal choices that include typical pet names such as 'Kitty cat', 'Junior', and 'Molly', as well as more unique identifiers like 'Monty', 'Hazelnut', and 'Taj'. This column may sometimes contain missing values, represented by 'nan', indicating the absence of a name, which might occur if the animal is unidentified or if the name was not recorded at the time of data entry. The presence of names such as 'Helo ( Hee-lo)' suggests that any included pronunciation guides or alternate spellings have also been documented in this column. Overall, the 'Name' column serves to provide a designation for each animal, which can be used for identification purposes and lends a personal aspect to the records in the dataset."]
                  Candidate Table Name: cultural-spaces.csv
                  Candidate Table Description: The table ""cultural-spaces.csv"" contains data from 2017 detailing various cultural spaces in Vancouver, providing information on their names, types, primary uses, ownership types, and active status along with their locations denoted through addresses, local areas, and geographical coordinates.
                  Candidate Table Column Names: [YEAR, CULTURAL_SPACE_NAME, ADDRESS, LOCAL_AREA, Geom]
                  Candidate Table Column Descriptions: ["The ""YEAR"" column in the given table represents the specific calendar year in which the cultural spaces are being referenced or documented. All the entries in this column have the value ""2017,"" indicating that each cultural space's data pertains to the year 2017. This column provides temporal context for the information across different cultural spaces, allowing for analysis or comparison of data within that particular year. It is a straightforward numerical representation of the year, serving as a timestamp or a temporal anchor for the dataset.", "The ""CULTURAL_SPACE_NAME"" column in the given table encompasses the official names of various cultural spaces located within Vancouver. Each entry in this column represents a distinct venue or institution that contributes to the cultural tapestry of the city. The diversity of cultural spaces captured by this column is vast, covering museums, galleries, studios, theatres, community centers, and educational institutions. These names serve as identifiers for these spaces, reflecting their primary use or the organization they host. This identifier is crucial for distinguishing between different types of cultural spaces, which can vary significantly in form and function, such as ""221A Artist Run Centre"" known for artistic exhibitions, ""Anza Club (Australia-New Zealand Association)"" which is a performance space, and ""Britannia Community Services Centre Society,"" a community center. The column plays a foundational role in understanding and mapping out the cultural infrastructure within the city, aiding in organizing, searching, and referencing the spaces listed in Vancouver's cultural landscape.", "The ""ADDRESS"" column provides the precise location details for each cultural space listed in the dataset. It includes the street address, city, and occasionally the postal code, ensuring a comprehensive localization of the establishment within Vancouver, British Columbia. The information is formatted in a manner typical to addresses, often detailing elements such as street number, street name, and additional area identifiers. This crucial data component allows for the identification and geographic positioning of diverse cultural spaces ranging from museums and galleries to community centers and artist studios across various local areas of Vancouver.", "The ""LOCAL_AREA"" column in the table represents the specific geographic region or neighborhood within Vancouver, British Columbia, where each cultural or community space is located. This column provides key insight into the spatial distribution of these sites across the city. The values in the column are names of neighborhoods such as ""Kitsilano,"" ""Strathcona,"" ""Downtown,"" ""UBC,"" ""Mount Pleasant,"" ""Grandview-Woodland,"" ""Fairview,"" ""Kensington-Cedar Cottage,"" ""South Cambie,"" ""Dunbar-Southlands,"" ""West End,"" and others, identifying the part of Vancouver each facility or organization is situated in. These designations help in categorizing the institutions based on their local context within the city's landscape, offering information that could be used for urban planning, community engagement, or demographic research. The presence of specific districts provides an understanding of the distribution and accessibility of cultural and educational spaces across the different urban contexts and community areas within Vancouver.", "The ""Geom"" column in the table contains spatial information about cultural spaces in Vancouver in the year 2017. Each entry in this column is represented as a string in JSON format, which specifies the type as ""Point"" along with a pair of ""coordinates"". These coordinates are two numeric values indicating the geographic location in terms of longitude and latitude respectively. This geographic information allows for the precise plotting of the cultural space's location on a map, providing a spatial dimension to the data. Since all the values share this format, the column serves as a geographic reference for each entry, facilitating spatial analysis or visualization in a geographical information system (GIS)."]
                  Answer: {"table_joinable": "No", "column_joinable" : "No", "target_match" : "null", "candidate_match": "null"}

            Example 03:
                  Target Table Name: community-centres.csv
                  Target Table Description: The table ""community-centres.csv"" provides detailed information about various community centres in Vancouver, listing their names, addresses, URLs for more information, geographical coordinates, and their respective local areas.
                  Target Table Column Names: [Geo Local Area]
                  Target Table Column Descriptions: ["The ""NAME"" column in the table represents the official titles or identities of various community centers and public facilities located across Vancouver. Each entry in this column provides the designated name of a specific public amenity, which serves as a key identifier for that location. These names are typically the most commonly recognized and used titles by the community and within administrative contexts. They might denote either the geographical area they serve, such as ""Dunbar"" or ""West Point Grey,"" or certain significant structures or features associated with the facility, such as ""Kitsilano War Memorial"" or ""Roundhouse."" It helps users to quickly recognize and reference a particular community location within the city.", "The ""ADDRESS"" column in the table contains the physical street addresses of various community centers or facilities located in Vancouver. Each address corresponds to a particular center, providing essential location information necessary for individuals to physically locate these sites. The addresses typically consist of a street number followed by the street name, which helps in pinpointing exact locations within the city’s extensive grid of streets. For example, an address like ""4747 Dunbar St"" specifies both the street number and the street name, making it useful for navigation or mailing purposes. The addresses vary in format slightly, reflecting the diversity of the city's street naming conventions, but consistently provide crucial geographical context for users seeking to visit these centers either for recreational or community services.", "The ""Geom"" column in the table contains geometric data represented in the GeoJSON format, specifically as ""Point"" features. Each entry in this column provides a set of coordinates—longitude and latitude—in an array format, indicating the geographic location of a specific site, typically a community center, within Vancouver. The coordinates are essential for mapping and spatial analysis, enabling the precise determination of each site's location on a map. This column helps in pinpointing the exact position of the facilities listed, aligning with their addresses provided in another column. The data is structured in a standardized way to ensure compatibility with geographic information systems (GIS) and other mapping tools, allowing for visualization and further spatial operations.", "The ""Geo Local Area"" column represents the neighborhood or district in which each entity, generally a community center or other public facility, is situated within a specific geographic context. This column provides a categorical depiction of the location, grouping facilities by the broader geographic region or locality in which they are based, often corresponding to known neighborhoods or administrative areas. The values are identifiers for regions within a city, such as Dunbar-Southlands, Hastings-Sunrise, Downtown, and others like Kitsilano or Strathcona. These identifiers align locations to well-recognized local communities, allowing users to associate facilities with their respective neighborhoods. This helps in understanding the regional distribution and accessibility of facilities, aiding in geographic analysis and resource allocation."]
                  Candidate Table Name: cultural-spaces.csv
                  Candidate Table Description: The table ""cultural-spaces.csv"" contains data from 2017 detailing various cultural spaces in Vancouver, providing information on their names, types, primary uses, ownership types, and active status along with their locations denoted through addresses, local areas, and geographical coordinates.
                  Candidate Table Column Names: [YEAR, CULTURAL_SPACE_NAME, ADDRESS, LOCAL_AREA, Geom]
                  Candidate Table Column Descriptions: ["The ""YEAR"" column in the given table represents the specific calendar year in which the cultural spaces are being referenced or documented. All the entries in this column have the value ""2017,"" indicating that each cultural space's data pertains to the year 2017. This column provides temporal context for the information across different cultural spaces, allowing for analysis or comparison of data within that particular year. It is a straightforward numerical representation of the year, serving as a timestamp or a temporal anchor for the dataset.", "The ""CULTURAL_SPACE_NAME"" column in the given table encompasses the official names of various cultural spaces located within Vancouver. Each entry in this column represents a distinct venue or institution that contributes to the cultural tapestry of the city. The diversity of cultural spaces captured by this column is vast, covering museums, galleries, studios, theatres, community centers, and educational institutions. These names serve as identifiers for these spaces, reflecting their primary use or the organization they host. This identifier is crucial for distinguishing between different types of cultural spaces, which can vary significantly in form and function, such as ""221A Artist Run Centre"" known for artistic exhibitions, ""Anza Club (Australia-New Zealand Association)"" which is a performance space, and ""Britannia Community Services Centre Society,"" a community center. The column plays a foundational role in understanding and mapping out the cultural infrastructure within the city, aiding in organizing, searching, and referencing the spaces listed in Vancouver's cultural landscape.", "The ""ADDRESS"" column provides the precise location details for each cultural space listed in the dataset. It includes the street address, city, and occasionally the postal code, ensuring a comprehensive localization of the establishment within Vancouver, British Columbia. The information is formatted in a manner typical to addresses, often detailing elements such as street number, street name, and additional area identifiers. This crucial data component allows for the identification and geographic positioning of diverse cultural spaces ranging from museums and galleries to community centers and artist studios across various local areas of Vancouver.", "The ""LOCAL_AREA"" column in the table represents the specific geographic region or neighborhood within Vancouver, British Columbia, where each cultural or community space is located. This column provides key insight into the spatial distribution of these sites across the city. The values in the column are names of neighborhoods such as ""Kitsilano,"" ""Strathcona,"" ""Downtown,"" ""UBC,"" ""Mount Pleasant,"" ""Grandview-Woodland,"" ""Fairview,"" ""Kensington-Cedar Cottage,"" ""South Cambie,"" ""Dunbar-Southlands,"" ""West End,"" and others, identifying the part of Vancouver each facility or organization is situated in. These designations help in categorizing the institutions based on their local context within the city's landscape, offering information that could be used for urban planning, community engagement, or demographic research. The presence of specific districts provides an understanding of the distribution and accessibility of cultural and educational spaces across the different urban contexts and community areas within Vancouver.", "The ""Geom"" column in the table contains spatial information about cultural spaces in Vancouver in the year 2017. Each entry in this column is represented as a string in JSON format, which specifies the type as ""Point"" along with a pair of ""coordinates"". These coordinates are two numeric values indicating the geographic location in terms of longitude and latitude respectively. This geographic information allows for the precise plotting of the cultural space's location on a map, providing a spatial dimension to the data. Since all the values share this format, the column serves as a geographic reference for each entry, facilitating spatial analysis or visualization in a geographical information system (GIS)."]
                  Answer: {"table_joinable": "Yes", "column_joinable" : "Yes", "target_match" : "Geo Local Area", "candidate_match": "LOCAL_AREA"}

            Example 04:
                  Target Table Name: rental-standards-current-issues.csv
                  Target Table Description: The ""rental-standards-current-issues.csv"" table provides a list of various business operators managing rental properties in Vancouver, detailing specific property issues, including the business operator's name, the URL for more details, the address with street number and name, the total number of outstanding issues, the total units within each property, the geographical coordinates, and the respective local area within the city.
                  Target Table Column Names: [Geo Local Area]
                  Target Table Column Descriptions: ["The ""Geom"" column contains geographical data formatted as a JSON string, which represents the spatial location of each business entity described in the table. Specifically, this data is structured as a GeoJSON 'Point' object, indicating the precise geographical coordinates of the business's physical location. Each value within this column includes a ""type"" key, which is always set to ""Point,"" and a ""coordinates"" key, which is an array containing two numerical values: the longitude and latitude of the location in decimal degrees. These coordinates allow for the mapping of these businesses onto a geographic information system (GIS) for spatial analysis or visualization purposes. In some instances, the ""Geom"" field contains a 'nan' value, indicating a missing or unavailable coordinate for that particular entry.", "The ""Geo Local Area"" column in the provided table represents the specific geographic region or neighborhood in which the property or entity is located. Each entry in this column corresponds to a broader area or locality within a city, helping to categorize the properties based on their general location. The values within this column include names such as ""Downtown,"" ""Grandview-Woodland,"" ""Strathcona,"" ""Kitsilano,"" ""South Cambie,"" ""Kerrisdale,"" ""West End,"" ""Dunbar-Southlands,"" and ""Fairview."" These names denote well-known local areas, likely used for urban planning or real estate purposes, and help users of the dataset quickly understand the spatial distribution of properties relative to recognized city districts or neighborhoods. Having this geographical context can be crucial for various analyses, such as demographic studies, real estate valuation, urban development planning, and customer segmentation based on location."]
                  Candidate Table Name: community-centres.csv
                  Candidate Table Description: The table ""community-centres.csv"" provides detailed information about various community centres in Vancouver, listing their names, addresses, URLs for more information, geographical coordinates, and their respective local areas.
                  Candidate Table Column Names: [NAME, ADDRESS, Geom, Geo Local Area]
                  Candidate Table Column Descriptions: ["The ""NAME"" column in the table represents the official titles or identities of various community centers and public facilities located across Vancouver. Each entry in this column provides the designated name of a specific public amenity, which serves as a key identifier for that location. These names are typically the most commonly recognized and used titles by the community and within administrative contexts. They might denote either the geographical area they serve, such as ""Dunbar"" or ""West Point Grey,"" or certain significant structures or features associated with the facility, such as ""Kitsilano War Memorial"" or ""Roundhouse."" It helps users to quickly recognize and reference a particular community location within the city.", "The ""ADDRESS"" column in the table contains the physical street addresses of various community centers or facilities located in Vancouver. Each address corresponds to a particular center, providing essential location information necessary for individuals to physically locate these sites. The addresses typically consist of a street number followed by the street name, which helps in pinpointing exact locations within the city’s extensive grid of streets. For example, an address like ""4747 Dunbar St"" specifies both the street number and the street name, making it useful for navigation or mailing purposes. The addresses vary in format slightly, reflecting the diversity of the city's street naming conventions, but consistently provide crucial geographical context for users seeking to visit these centers either for recreational or community services.", "The ""Geom"" column in the table contains geometric data represented in the GeoJSON format, specifically as ""Point"" features. Each entry in this column provides a set of coordinates—longitude and latitude—in an array format, indicating the geographic location of a specific site, typically a community center, within Vancouver. The coordinates are essential for mapping and spatial analysis, enabling the precise determination of each site's location on a map. This column helps in pinpointing the exact position of the facilities listed, aligning with their addresses provided in another column. The data is structured in a standardized way to ensure compatibility with geographic information systems (GIS) and other mapping tools, allowing for visualization and further spatial operations.", "The ""Geo Local Area"" column represents the neighborhood or district in which each entity, generally a community center or other public facility, is situated within a specific geographic context. This column provides a categorical depiction of the location, grouping facilities by the broader geographic region or locality in which they are based, often corresponding to known neighborhoods or administrative areas. The values are identifiers for regions within a city, such as Dunbar-Southlands, Hastings-Sunrise, Downtown, and others like Kitsilano or Strathcona. These identifiers align locations to well-recognized local communities, allowing users to associate facilities with their respective neighborhoods. This helps in understanding the regional distribution and accessibility of facilities, aiding in geographic analysis and resource allocation."]
                  Answer: {"table_joinable": "Yes", "column_joinable" : "Yes", "target_match" : "Geo Local Area", "candidate_match": "Geo Local Area"}
    """
    user_msg = f'''
                  Target Table Name: {query_table_name}
                  Target Table Description: {query_table_description}
                  Target Table Column Names: {query_table_column_names}
                  Target Table Column Descriptions: {query_table_column_descriptions}
                  Candidate Table Name: {candidate_table_name}
                  Candidate Table Description: {candidate_table_description}
                  Candidate Table Column Names: {candidate_table_column_names}
                  Candidate Table Column Descriptions: {candidate_table_column_descriptions}
                  Answer:'''.strip()

    return system_msg, user_msg

In [ ]:
BENCHMARK = 'NextiaJD-S' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider
column_descriptions = pd.read_csv(f'{BENCHMARK}/column_descriptions.csv')
column_descriptions.head()

In [ ]:
table_descriptions = pd.read_csv(f'{BENCHMARK}/table_descriptions.csv')
table_descriptions.head()

In [ ]:
table_cartesians = pd.read_csv(f'{BENCHMARK}/table_cartesians.csv')
table_cartesians.head()

In [ ]:
prompts = {}

for i in tqdm(range(len(table_cartesians))):

    query_table_name = table_cartesians.iloc[i, 0]
    candidate_table_name = table_cartesians.iloc[i, 1]
    prompts.setdefault(query_table_name, {})

    query_table_description = table_descriptions[table_descriptions['TableName'] == query_table_name].iloc[0,1]
    query_table_column_names = column_descriptions[column_descriptions['TableName'] == query_table_name].iloc[:,1].values
    query_table_column_descriptions = column_descriptions[column_descriptions['TableName'] == query_table_name].iloc[:,2].values
        
    candidate_table_description = table_descriptions[table_descriptions['TableName'] == candidate_table_name].iloc[0,1]
    candidate_table_column_names = column_descriptions[column_descriptions['TableName'] == candidate_table_name].iloc[:,1].values
    candidate_table_column_descriptions = column_descriptions[column_descriptions['TableName'] == candidate_table_name].iloc[:,2].values
        
    system_msg, user_msg = generate_prompt_JD_prediction(query_table_name, query_table_description, query_table_column_names, query_table_column_descriptions,
                                            candidate_table_name, candidate_table_description, candidate_table_column_names, candidate_table_column_descriptions)
  
    prompts[query_table_name][candidate_table_name] = {
        'system_msg': system_msg,
        'user_msg': user_msg,
    }

In [ ]:
with open(f'{BENCHMARK}/prompts/table_JD.json', 'w') as f:
    json.dump(prompts, f)